# Fixation Elman mRNN Smoke Test

This notebook is the minimal end-to-end workflow for fitting the fixation mRNN to two target spaces:

1. Raw mean firing-rate time series.
2. Per-region firing-rate PCs.

The code uses a fixed region order from `configs/ephys_fixation_mrnn.yaml`. Output dimensions are inferred from the target tensors, so raw firing-rate models read out the number of units in each region and PC models read out the retained PC dimensions for each region.

## 1. Setup

Use the local repository code and load the compact modeling API.

In [ ]:
from pathlib import Path
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from dal_monte_2022_analysis.ephys.modeling import (
    extract_region_currents,
    load_fixation_mrnn_config,
    make_targets,
    output_pc_scores,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    settings_from_config,
    train_fixation_mrnn_scratch,
    variance_comparison,
)


def display_figure(fig, *, dpi=90):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))


def plot_loss(history, title):
    fig, ax = plt.subplots(figsize=(6, 3.2), dpi=130)
    ax.plot(history["iteration"], history["loss"], label="total")
    ax.plot(history["iteration"], history["mse_loss"], label="MSE")
    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    display_figure(fig)
    return fig, ax


def plot_reconstructions(replay, title, max_features=2):
    checkpoint = replay["checkpoint"]
    regions = tuple(replay["region_order"])
    conditions = tuple(replay["condition_order"])
    time = np.asarray(checkpoint["timeline_s"], dtype=float)
    fig, axes = plt.subplots(len(regions), max_features, figsize=(4.2 * max_features, 2.1 * len(regions)), dpi=130, squeeze=False)
    for row, region in enumerate(regions):
        observed = np.asarray(checkpoint["target_by_region"][region], dtype=float)
        predicted = replay["output_by_region"][region].detach().cpu().numpy()
        feature_names = checkpoint["features_by_region"][region]
        for feature_idx in range(max_features):
            ax = axes[row, feature_idx]
            if feature_idx >= observed.shape[-1]:
                ax.axis("off")
                continue
            for cond_idx, condition in enumerate(conditions):
                ax.plot(time, observed[cond_idx, :, feature_idx], linewidth=1.5, label=f"{condition} observed")
                ax.plot(time, predicted[cond_idx, :, feature_idx], linestyle="--", linewidth=1.2, label=f"{condition} mRNN")
            ax.set_title(f"{region}: {feature_names[feature_idx]}", fontsize=9)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Target")
            ax.grid(alpha=0.2)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=7)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    display_figure(fig)
    return fig, axes


def plot_output_pcs(replay, title):
    regions = tuple(replay["region_order"])
    conditions = tuple(replay["condition_order"])
    time = np.asarray(replay["checkpoint"]["timeline_s"], dtype=float)
    fig, axes = plt.subplots(len(regions), len(conditions), figsize=(11, 7), dpi=130, squeeze=False, sharex=True)
    for row, region in enumerate(regions):
        scores = output_pc_scores(replay, region=region, n_components=3)
        for col, condition in enumerate(conditions):
            ax = axes[row, col]
            cond_idx = conditions.index(condition)
            for pc_idx, linestyle in enumerate(("-", "--", ":")):
                ax.plot(time, scores[cond_idx, :, pc_idx], linestyle=linestyle, label=f"PC{pc_idx + 1}")
            if row == 0:
                ax.set_title(condition)
            if col == 0:
                ax.set_ylabel(f"{region}\nscore")
            if row == len(regions) - 1:
                ax.set_xlabel("Time (s)")
    axes[0, -1].legend(frameon=False, fontsize=7)
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    display_figure(fig)
    return fig, axes


def summarize_replay(replay):
    current_df, current_vectors = extract_region_currents(replay)
    display(reconstruction_accuracy(replay))
    display(variance_comparison(replay))
    display(current_df.groupby(["target_region", "source_region"], as_index=False)["relative_contribution"].mean())
    return current_df, current_vectors

## 2. Model Definition and Equations

The model is an Elman mRNN. For condition input $u_t \in \mathbb{R}^3$, hidden activity $h_t$, and region-specific readout $\hat y^r_t$, the forward pass is:

$$h_{t+1}=\phi(W_{rec}h_t + W_{in}u_t + b + \epsilon_t)$$

$$\hat y^r_t = W^r_{out}h^r_t + b^r_{out}$$

There is no separate latent/preactivation state in this simplified pipeline. The model only stores the Elman hidden sequence and per-region readout outputs.

Current extraction in `extract_region_currents` uses the effective recurrent matrix after masks and constraints. For target region $r$ and source region $s$, the function slices the recurrent block $W_{rec}^{r,s}$ and multiplies it by the previous hidden activity of the source region:

$$I_{s\to r}(t)=W_{rec}^{r,s}h^s_{t-1}$$

The relative contribution plotted and tabulated below is:

$$C_{s\to r}(t)=\frac{\|I_{s\to r}(t)\|_2}{\sum_q \|I_{q\to r}(t)\|_2}$$

PCA trajectories and downstream summaries in this notebook are computed from the model readout outputs, not hidden units.

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs" / "ephys_fixation_mrnn.yaml")
settings = settings_from_config(cfg)
settings.dataset_cfg_path = str(repo_root / "configs" / "dataset.yaml")
settings.device = "auto"
settings.hidden_units = 8
settings.epochs = 10_000
settings.initialization_mode = "single"
settings.overwrite_seed_plan = False
settings

## 3. Targets

Targets are built once from the combined fixation PSTH export. Raw firing rates are normalized with one global robust scale across all regions, units, fixation types, and time bins before either raw targets or region PCs are used for fitting.

In [ ]:
targets = make_targets(settings)
rows = []
for region in targets.region_order:
    rows.append({
        "region": region,
        "raw_shape": targets.raw_by_region[region].shape,
        "pc_shape": targets.pcs_by_region[region].shape,
        "raw_variance": float(np.var(targets.raw_by_region[region])),
        "pc_variance": float(np.var(targets.pcs_by_region[region])),
    })
print("normalization scale:", targets.normalization_scale)
display(pd.DataFrame(rows))

## 4. Raw Firing-Rate Fit

This section fits the model readouts directly to normalized mean firing-rate traces. The output dimension for each region is the number of recorded units in that region.

In [ ]:
raw_settings = settings_from_config(cfg)
raw_settings.dataset_cfg_path = settings.dataset_cfg_path
raw_settings.device = settings.device
raw_settings.hidden_units = settings.hidden_units
raw_settings.epochs = settings.epochs
raw_settings.initialization_mode = "single"
raw_settings.target_mode = "raw_fr"
raw_settings.seed = 123456

raw_result = train_fixation_mrnn_scratch(raw_settings, scratch_id="smoke_raw_fr", overwrite=True)
plot_loss(raw_result["history"], "Raw Firing-Rate Fit")
raw_replay = replay_fixation_mrnn_run(raw_result["run_dir"], device="cpu")
raw_current_df, raw_current_vectors = summarize_replay(raw_replay)

In [ ]:
plot_reconstructions(raw_replay, "Raw Firing-Rate Targets and Reconstructions", max_features=2)
plot_output_pcs(raw_replay, "Raw Firing-Rate Model Output PCs")

## 5. Firing-Rate PC Fit

This section fits the same Elman mRNN architecture to per-region PCs. PC dimensions are inferred from the PCA threshold and can differ by region.

In [ ]:
pc_settings = settings_from_config(cfg)
pc_settings.dataset_cfg_path = settings.dataset_cfg_path
pc_settings.device = settings.device
pc_settings.hidden_units = settings.hidden_units
pc_settings.epochs = settings.epochs
pc_settings.initialization_mode = "single"
pc_settings.target_mode = "region_pcs"
pc_settings.seed = 223456

pc_result = train_fixation_mrnn_scratch(pc_settings, scratch_id="smoke_region_pcs", overwrite=True)
plot_loss(pc_result["history"], "Firing-Rate PC Fit")
pc_replay = replay_fixation_mrnn_run(pc_result["run_dir"], device="cpu")
pc_current_df, pc_current_vectors = summarize_replay(pc_replay)

In [ ]:
plot_reconstructions(pc_replay, "Firing-Rate PC Targets and Reconstructions", max_features=2)
plot_output_pcs(pc_replay, "Firing-Rate PC Model Output PCs")

## 6. Multiple Initializations

To run multiple initializations, set `initialization_mode = "multiple"` and choose `n_initializations`. The seed list is written to `seed_plan.json` in the scratch output directory and reused on reruns unless `overwrite_seed_plan` is set to `True`.

In [ ]:
# multi_settings = settings_from_config(cfg)
# multi_settings.dataset_cfg_path = settings.dataset_cfg_path
# multi_settings.target_mode = "raw_fr"
# multi_settings.initialization_mode = "multiple"
# multi_settings.n_initializations = 100
# multi_settings.overwrite_seed_plan = False
# multi_result = train_fixation_mrnn_scratch(multi_settings, scratch_id="raw_fr_100init", overwrite=True)